# BYOL — Bootstrap Your Own Latent
**Grill et al., NeurIPS 2020**

**Category:** `03-Self-Supervised-Learning / 03-FeaturePrediction`

BYOL **discards contrastive learning entirely** — no negatives needed. Instead, an **online network** predicts the representation of a **target (teacher) network**, which is updated by EMA.

| | SimCLR / MoCo | BYOL |
|---|---|---|
| Needs negatives? | Yes | **No** |
| Collapse prevention | Repelling negatives | Asymmetric architecture + EMA |
| Extra module | — | **Predictor MLP** on online only |

> **How does it not collapse?** The predictor is asymmetric (only online has it) + target moves slowly via EMA → provides a stable, non-trivial target that the online network must chase.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
import copy
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


## Step 1: Augmentation

In [ ]:
class BYOLAugmentation:
    def __init__(self, size=32):
        self.transform = T.Compose([
            T.RandomResizedCrop(size, scale=(0.2, 1.0)),
            T.RandomHorizontalFlip(),
            T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            T.RandomGrayscale(p=0.2),
            T.GaussianBlur(kernel_size=3),
            T.ToTensor(),
            T.Normalize([0.4914, 0.4822, 0.4465], [0.2023, 0.1994, 0.2010]),
        ])
    def __call__(self, x):
        return self.transform(x), self.transform(x)

class TwoViewDataset(Dataset):
    def __init__(self, aug):
        self.base = torchvision.datasets.CIFAR10('./data', True, download=True, transform=None)
        self.aug  = aug
    def __len__(self): return len(self.base)
    def __getitem__(self, idx):
        img, _ = self.base[idx]; return self.aug(img)


## Step 2: BYOL Network

```
Online:  x → f_θ (encoder) → g_θ (projector) → q_θ (predictor) → p
Target:  x → f_ξ (encoder) → g_ξ (projector)                   → z'  (stop-grad)
Loss: MSE(normalize(p), normalize(z'))
```

**Target network** = EMA of online: `ξ ← τ·ξ + (1−τ)·θ`  
Target has **no predictor** — this asymmetry prevents collapse.


In [ ]:
def make_mlp(in_dim, hidden_dim, out_dim):
    return nn.Sequential(
        nn.Linear(in_dim, hidden_dim),
        nn.BatchNorm1d(hidden_dim),
        nn.ReLU(inplace=True),
        nn.Linear(hidden_dim, out_dim),
    )


class BYOL(nn.Module):
    def __init__(self, base_encoder=torchvision.models.resnet18,
                 proj_dim=256, pred_dim=128, tau=0.996):
        super().__init__()
        self.tau = tau  # EMA momentum

        # Online: encoder + projector + predictor
        enc = base_encoder(weights=None)
        feat_dim = enc.fc.in_features           # 512
        enc.fc = nn.Identity()
        self.online_enc  = enc
        self.online_proj = make_mlp(feat_dim, 4096, proj_dim)
        self.predictor   = make_mlp(proj_dim,  4096, pred_dim)

        # Target: encoder + projector only (no predictor, no grad)
        self.target_enc  = copy.deepcopy(self.online_enc)
        self.target_proj = copy.deepcopy(self.online_proj)
        for p in list(self.target_enc.parameters()) + list(self.target_proj.parameters()):
            p.requires_grad = False

    @torch.no_grad()
    def _ema_update(self):
        for p_o, p_t in zip(self.online_enc.parameters(), self.target_enc.parameters()):
            p_t.data = self.tau * p_t.data + (1 - self.tau) * p_o.data
        for p_o, p_t in zip(self.online_proj.parameters(), self.target_proj.parameters()):
            p_t.data = self.tau * p_t.data + (1 - self.tau) * p_o.data

    def _online(self, x):
        h = self.online_enc(x)
        z = self.online_proj(h)
        p = self.predictor(z)
        return p

    @torch.no_grad()
    def _target(self, x):
        h = self.target_enc(x)
        z = self.target_proj(h)
        return z

    def forward(self, x1, x2):
        # Online predicts target's projection of the other view
        p1 = self._online(x1)
        p2 = self._online(x2)
        with torch.no_grad():
            z1 = self._target(x1)
            z2 = self._target(x2)

        # Symmetric loss
        loss = self._loss(p1, z2) + self._loss(p2, z1)
        self._ema_update()
        return loss.mean()

    @staticmethod
    def _loss(p, z):
        """MSE on L2-normalised vectors (equivalent to 2 - 2·cos_sim)."""
        p = F.normalize(p, dim=-1)
        z = F.normalize(z, dim=-1)
        return 2 - 2 * (p * z).sum(dim=-1)


## Step 3: Training

In [ ]:
BATCH_SIZE  = 128
EPOCHS      = 10
LR          = 3e-4

aug    = BYOLAugmentation()
loader = DataLoader(TwoViewDataset(aug), batch_size=BATCH_SIZE,
                    shuffle=True, num_workers=2, drop_last=True)

model     = BYOL().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1.5e-6)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

losses = []
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = []
    for (x1, x2) in loader:
        x1, x2 = x1.to(device), x2.to(device)
        loss = model(x1, x2)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss.append(loss.item())
    scheduler.step()
    avg = sum(epoch_loss)/len(epoch_loss)
    losses.append(avg)
    print(f'Epoch {epoch+1:2d}/{EPOCHS} | Loss: {avg:.4f}')

import os; os.makedirs('saved', exist_ok=True)
torch.save(model.state_dict(), 'saved/byol.pt')


## Step 4: Linear Evaluation

In [ ]:
model.load_state_dict(torch.load('saved/byol.pt', map_location=device))
for p in model.online_enc.parameters():
    p.requires_grad = False

linear = nn.Linear(512, 10).to(device)
opt_lin = torch.optim.Adam(linear.parameters(), lr=1e-3)

eval_tf = T.Compose([T.ToTensor(),
    T.Normalize([0.4914,0.4822,0.4465],[0.2023,0.1994,0.2010])])
train_ld = DataLoader(torchvision.datasets.CIFAR10('./data', True,  eval_tf, download=True), 256, shuffle=True)
test_ld  = DataLoader(torchvision.datasets.CIFAR10('./data', False, eval_tf, download=True), 256)

for epoch in range(5):
    model.eval(); linear.train()
    for imgs, labels in train_ld:
        imgs, labels = imgs.to(device), labels.to(device)
        with torch.no_grad():
            h = model.online_enc(imgs)
        loss = F.cross_entropy(linear(h), labels)
        opt_lin.zero_grad(); loss.backward(); opt_lin.step()

correct = total = 0
with torch.no_grad():
    for imgs, labels in test_ld:
        imgs, labels = imgs.to(device), labels.to(device)
        h = model.online_enc(imgs)
        preds = linear(h).argmax(1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
print(f'BYOL Linear Eval: {correct/total*100:.2f}%')
